In [ ]:
# In[1]:

import json

# Load the task.json file to understand the alert context
task_json_path = 'rca100/cases/t024/task.json'
with open(task_json_path, 'r') as file:
    task_data = json.load(file)

# Extract relevant information from the task data
alert_title = task_data.get('alert_title')
alert_window = task_data.get('alert_window')
alert_entity = task_data.get('alert_entity')
prompt_text = task_data.get('prompt_text')

alert_title, alert_window, alert_entity, prompt_text

```
Out[1]:
```
The alert is related to the `inventory::/api/v1/inventory/{productId}` operation in the APM domain. The alert was triggered due to a sudden increase in error counts, as indicated by the rule name "inventory错误次数突增". The alert time window is from `2026-04-04T22:37:09.273091+08:00` to `2026-04-04T22:45:01+08:00`. The user is requesting an analysis of the root cause.

The original code execution output of IPython Kernel is also provided below for reference:

('', {'start': '2026-04-04T22:37:09.273091+08:00', 'end': '2026-04-04T22:45:01+08:00'}, {'entity_id': '162799adab6799326676266a2c96cf3e', 'entity_name': 'inventory::/api/v1/inventory/{productId}', 'entity_type': 'apm.operation', 'entity_domain': 'apm'}, '<code vibeops_object type="alert_event">inventory错误次数突增<alert_event event_id="f117cea2bc1c5c47816e78c6fd7573c2" trans_id="sfhu5c5e6t7sfeo75j5he606o5" rule_id="663b6c72-c01c-49e2-b1a7-805ae65925b3" rule_name="inventory错误次数突增" /></code>\n\n帮我分析下根因。')```
```

In [ ]:
# In[2]:

import pandas as pd

# Load the metrics.parquet file
metrics_path = 'rca100/cases/t024/metrics.parquet'
metrics_df = pd.read_parquet(metrics_path)

# Extract all available KPIs and their corresponding components
available_kpis = metrics_df[['entity_set', 'entity_name', 'metric']].drop_duplicates()

available_kpis

```
Out[2]:
```
The `metrics.parquet` file contains KPIs across various components, including Kubernetes nodes (`k8s.node`) and APM instances (`apm.instance`). Key metrics include `node_not_ready_status`, `node_ready_status`, `node_disk_usage_rate`, `node_memory_usage_rate`, `node_pod_running_count`, and `error_rate`. These KPIs are associated with different entities such as nodes and specific service instances like `frontend` and `inventory`. For a comprehensive analysis, we can focus on metrics like `error_rate` for APM instances and resource usage metrics for Kubernetes nodes.

The original code execution output of IPython Kernel is also provided below for reference:

entity_set                      entity_name                  metric
0          k8s.node                                    node_not_ready_status
2100       k8s.node                                        node_ready_status
4200       k8s.node                                     node_disk_usage_rate
4760       k8s.node                                   node_memory_usage_rate
6300       k8s.node                                   node_pod_running_count
...             ...                              ...                     ...
88660  apm.instance        frontend-6df7d557f5-sgfcb              error_rate
88677  apm.instance       inventory-794d9f8f85-hhjtj              error_rate
88681  apm.instance        frontend-6df7d557f5-nz7g4              error_rate
88698  apm.instance  frontend-proxy-67d64dc4f8-8c99b              error_rate
88723  apm.instance        inventory-887d8db89-7v4s5              error_rate

[298 rows x 3 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

# Filter the metrics data for APM instances and the 'error_rate' metric
apm_error_rate_df = metrics_df[
    (metrics_df['entity_set'] == 'apm.instance') & (metrics_df['metric'] == 'error_rate')
]

# Ensure the 'value' column is numeric for aggregation
apm_error_rate_df['value'] = pd.to_numeric(apm_error_rate_df['value'], errors='coerce')

# Calculate the global P95 threshold for the 'error_rate' metric
error_rate_p95_threshold = apm_error_rate_df['value'].quantile(0.95)

error_rate_p95_threshold

```
Out[3]:
```
The global 95th percentile (P95) threshold for the `error_rate` metric across all APM instances is approximately 0.9144. This threshold can be used to identify anomalies in the `inventory` service during the alert time window.

The original code execution output of IPython Kernel is also provided below for reference:

0.914394910789974```
```

In [ ]:
# In[4]:

from datetime import datetime
import pytz

# Define the alert time window in UTC+8
alert_start = datetime.fromisoformat('2026-04-04T22:37:09.273091+08:00').astimezone(pytz.timezone('Asia/Shanghai'))
alert_end = datetime.fromisoformat('2026-04-04T22:45:01+08:00').astimezone(pytz.timezone('Asia/Shanghai'))

# Convert the 'time' column to numeric and filter for the 'inventory' service within the alert time window
apm_error_rate_df['time'] = pd.to_numeric(apm_error_rate_df['time'], errors='coerce')
inventory_error_rate_df = apm_error_rate_df[
    (apm_error_rate_df['entity_name'].str.contains('inventory', case=False)) &
    (apm_error_rate_df['time'] >= alert_start.timestamp() * 1_000_000) &
    (apm_error_rate_df['time'] <= alert_end.timestamp() * 1_000_000)
]

# Identify anomalies where the 'error_rate' exceeds the global P95 threshold
inventory_anomalies = inventory_error_rate_df[inventory_error_rate_df['value'] > error_rate_p95_threshold]

inventory_anomalies

```
Out[4]:
```
No anomalies were detected in the `error_rate` metric for the `inventory` service during the alert time window, as the `error_rate` did not exceed the global P95 threshold of 0.9144.

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [time, domain, entity_set, entity_id, entity_name, metric, value, metric_set_id, service]
Index: []```
```

In [ ]:
# In[5]:

# Load the traces.parquet file
traces_path = 'rca100/cases/t024/traces.parquet'
traces_df = pd.read_parquet(traces_path)

# Ensure relevant columns are numeric for filtering
traces_df['startTime'] = pd.to_numeric(traces_df['startTime'], errors='coerce')
traces_df['endTime'] = pd.to_numeric(traces_df['endTime'], errors='coerce')
traces_df['duration'] = pd.to_numeric(traces_df['duration'], errors='coerce')
traces_df['statusCode'] = pd.to_numeric(traces_df['statusCode'], errors='coerce')

# Filter traces related to the 'inventory::/api/v1/inventory/{productId}' operation within the alert time window
inventory_traces_df = traces_df[
    (traces_df['spanName'].str.contains('inventory::/api/v1/inventory/{productId}', case=False)) &
    (traces_df['startTime'] >= alert_start.timestamp() * 1_000_000_000) &
    (traces_df['endTime'] <= alert_end.timestamp() * 1_000_000_000)
]

# Look for traces with errors (statusCode != 0 or statusMessage indicating an error)
inventory_error_traces = inventory_traces_df[
    (inventory_traces_df['statusCode'] != 0) | 
    (inventory_traces_df['statusMessage'].str.contains('error', case=False, na=False))
]

inventory_error_traces

```
Out[5]:
```
No error traces were found for the `inventory::/api/v1/inventory/{productId}` operation within the alert time window. This indicates that there were no traces with `statusCode` errors or error-related `statusMessage` during this period.

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [traceId, spanId, parentSpanId, kind, spanName, startTime, endTime, duration, serviceName, pid, ip, hostname, statusCode, statusMessage, traceState, resources, attributes, __topic__, __tag__:__pack_id__, events, links]
Index: []

[0 rows x 21 columns]```
```

In [ ]:
# In[6]:

# Load the logs.parquet file
logs_path = 'rca100/cases/t024/logs.parquet'
logs_df = pd.read_parquet(logs_path)

# Ensure the '_time_' column is in datetime format
logs_df['_time_'] = pd.to_datetime(logs_df['_time_'], errors='coerce')

# Filter logs for the 'inventory' service within the alert time window
inventory_logs_df = logs_df[
    (logs_df['_time_'] >= alert_start) &
    (logs_df['_time_'] <= alert_end) &
    (logs_df['_container_name_'].str.contains('inventory', case=False, na=False))
]

# Look for error messages or unusual patterns in the log content
inventory_error_logs = inventory_logs_df[
    inventory_logs_df['content'].str.contains('error|exception|fail|critical', case=False, na=False)
]

inventory_error_logs

```
Out[6]:
```
No error messages or unusual patterns were found in the logs for the `inventory` service within the alert time window. This suggests that there were no logged errors or critical issues during this period.

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [content, _time_, _source_, _container_ip_, _image_name_, _container_name_, _pod_name_, _namespace_, _pod_uid_, __topic__, __tag__:__pack_id__, __tag__:__hostname__, __tag__:_node_name_, __tag__:_node_ip_, __tag__:_cluster_id_]
Index: []

[0 rows x 15 columns]```
```

In [ ]:
# In[7]:

# Load the topology.json file
topology_path = 'rca100/cases/t024/topology.json'
with open(topology_path, 'r') as file:
    topology_data = json.load(file)

# Extract entities and edges from the topology data
entities = topology_data.get('entities', [])
edges = topology_data.get('edges', [])

# Convert edges to a DataFrame for analysis
edges_df = pd.DataFrame(edges)

# Filter edges to identify upstream and downstream services related to the 'inventory' service
inventory_related_edges = edges_df[
    (edges_df['src_type'] == 'apm.service') & (edges_df['src'].str.contains('inventory', case=False)) |
    (edges_df['dst_type'] == 'apm.service') & (edges_df['dst'].str.contains('inventory', case=False))
]

inventory_related_edges

```
Out[7]:
```
The service dependency graph does not show any upstream or downstream relationships involving the `inventory` service. This indicates that no direct dependencies or interactions with other services were identified in the topology data.

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [src, src_type, dst, dst_type, relation, first_observed, last_observed]
Index: []```
```

In [ ]:
# In[8]:

# Filter node-level metrics ('k8s.node') within the alert time window
node_metrics_df = metrics_df[
    (metrics_df['entity_set'] == 'k8s.node') &
    (metrics_df['time'] >= alert_start.timestamp() * 1_000_000) &
    (metrics_df['time'] <= alert_end.timestamp() * 1_000_000)
]

# Focus on specific metrics: 'node_memory_usage_rate', 'node_cpu_usage_rate', 'node_disk_usage_rate'
node_resource_metrics = node_metrics_df[
    node_metrics_df['metric'].isin(['node_memory_usage_rate', 'node_cpu_usage_rate', 'node_disk_usage_rate'])
]

node_resource_metrics

```
Out[8]:
```
The node-level metrics within the alert time window include `node_disk_usage_rate`, `node_memory_usage_rate`, and `node_cpu_usage_rate`. These metrics show resource usage patterns for Kubernetes nodes. However, the data is extensive, and further analysis is needed to identify specific anomalies or resource constraints. If required, we can focus on specific metrics or nodes for a more detailed investigation.

The original code execution output of IPython Kernel is also provided below for reference:

time domain entity_set entity_id entity_name                metric      value                                     metric_set_id service
4321   1775313459000000    k8s   k8s.node                        node_disk_usage_rate  24.022831  k8s@metric_set@k8s.metric.high_level_metric_node    None
4322   1775313489000000    k8s   k8s.node                        node_disk_usage_rate  24.022921  k8s@metric_set@k8s.metric.high_level_metric_node    None
4323   1775313519000000    k8s   k8s.node                        node_disk_usage_rate  24.023014  k8s@metric_set@k8s.metric.high_level_metric_node    None
4324   1775313549000000    k8s   k8s.node                        node_disk_usage_rate  24.023112  k8s@metric_set@k8s.metric.high_level_metric_node    None
4325   1775313579000000    k8s   k8s.node                        node_disk_usage_rate  24.023212  k8s@metric_set@k8s.metric.high_level_metric_node    None
...                 ...    ...        ...       ...         ...                   ...        ...                                               ...     ...
12171  1775313759000000    k8s   k8s.node                         node_cpu_usage_rate   2.078596  k8s@metric_set@k8s.metric.high_level_metric_node    None
12172  1775313789000000    k8s   k8s.node                         node_cpu_usage_rate   2.052250  k8s@metric_set@k8s.metric.high_level_metric_node    None
12173  1775313819000000    k8s   k8s.node                         node_cpu_usage_rate   1.987509  k8s@metric_set@k8s.metric.high_level_metric_node    None
12174  1775313849000000    k8s   k8s.node                         node_cpu_usage_rate   2.046745  k8s@metric_set@k8s.metric.high_level_metric_node    None
12175  1775313879000000    k8s   k8s.node                         node_cpu_usage_rate   2.028682  k8s@metric_set@k8s.metric.high_level_metric_node    None

[630 rows x 9 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[9]:

# Filter node-level metrics for the entire dataset
all_node_metrics = metrics_df[
    (metrics_df['entity_set'] == 'k8s.node') &
    (metrics_df['metric'].isin(['node_memory_usage_rate', 'node_cpu_usage_rate', 'node_disk_usage_rate']))
]

# Ensure the 'value' column is numeric for aggregation
all_node_metrics['value'] = pd.to_numeric(all_node_metrics['value'], errors='coerce')

# Calculate the global P95 threshold for each metric
p95_thresholds = all_node_metrics.groupby('metric')['value'].quantile(0.95)

p95_thresholds

```
Out[9]:
```
The global 95th percentile (P95) thresholds for the node-level metrics are as follows:
- `node_cpu_usage_rate`: 7.9773
- `node_disk_usage_rate`: 34.8443
- `node_memory_usage_rate`: 91.3494

These thresholds can be used to identify anomalies in resource usage during the alert time window.

The original code execution output of IPython Kernel is also provided below for reference:

metric
node_cpu_usage_rate        7.977259
node_disk_usage_rate      34.844325
node_memory_usage_rate    91.349438
Name: value, dtype: float64```
```

In [ ]:
# In[10]:

# Filter node-level metrics within the alert time window
filtered_node_metrics = node_resource_metrics[
    (node_resource_metrics['metric'].isin(['node_memory_usage_rate', 'node_cpu_usage_rate', 'node_disk_usage_rate'])) &
    (node_resource_metrics['value'] > node_resource_metrics['metric'].map(p95_thresholds))
]

filtered_node_metrics

```
Out[10]:
```
During the alert time window, anomalies were detected in the node-level metrics. Specifically:
- `node_disk_usage_rate` exceeded its global P95 threshold of 34.8443.
- `node_cpu_usage_rate` exceeded its global P95 threshold of 7.9773.

These anomalies indicate potential resource constraints or high usage on Kubernetes nodes during the alert period. Further investigation into the affected nodes may be necessary.

The original code execution output of IPython Kernel is also provided below for reference:

time domain entity_set entity_id entity_name                metric      value                                     metric_set_id service
6281   1775313459000000    k8s   k8s.node                        node_disk_usage_rate  34.929127  k8s@metric_set@k8s.metric.high_level_metric_node    None
6282   1775313489000000    k8s   k8s.node                        node_disk_usage_rate  34.938129  k8s@metric_set@k8s.metric.high_level_metric_node    None
6283   1775313519000000    k8s   k8s.node                        node_disk_usage_rate  34.940305  k8s@metric_set@k8s.metric.high_level_metric_node    None
6284   1775313549000000    k8s   k8s.node                        node_disk_usage_rate  34.878347  k8s@metric_set@k8s.metric.high_level_metric_node    None
6285   1775313579000000    k8s   k8s.node                        node_disk_usage_rate  34.880461  k8s@metric_set@k8s.metric.high_level_metric_node    None
...                 ...    ...        ...       ...         ...                   ...        ...                                               ...     ...
12031  1775313759000000    k8s   k8s.node                         node_cpu_usage_rate  18.542337  k8s@metric_set@k8s.metric.high_level_metric_node    None
12032  1775313789000000    k8s   k8s.node                         node_cpu_usage_rate  18.391883  k8s@metric_set@k8s.metric.high_level_metric_node    None
12033  1775313819000000    k8s   k8s.node                         node_cpu_usage_rate  17.891875  k8s@metric_set@k8s.metric.high_level_metric_node    None
12034  1775313849000000    k8s   k8s.node                         node_cpu_usage_rate  18.972345  k8s@metric_set@k8s.metric.high_level_metric_node    None
12035  1775313879000000    k8s   k8s.node                         node_cpu_usage_rate  18.167012  k8s@metric_set@k8s.metric.high_level_metric_node    None

[70 rows x 9 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[11]:

# Group anomalies by node and metric to calculate the deviation from the P95 thresholds
filtered_node_metrics['deviation'] = filtered_node_metrics.apply(
    lambda row: row['value'] - p95_thresholds[row['metric']], axis=1
)

# Identify the node with the most significant deviation for each metric
most_significant_deviation = filtered_node_metrics.groupby(['entity_name', 'metric'])['deviation'].max().reset_index()

most_significant_deviation

```
Out[11]:
```
The most significant deviations from the global P95 thresholds during the alert time window occurred in the following metrics:
- `node_cpu_usage_rate`: Deviation of 10.9951, indicating a substantial CPU usage anomaly.
- `node_disk_usage_rate`: Deviation of 0.0960, indicating a minor disk usage anomaly.
- `node_memory_usage_rate`: Deviation of 0.3463, indicating a small memory usage anomaly.

The most critical issue is the high CPU usage, which significantly exceeded the threshold. Further investigation should focus on the node experiencing this CPU anomaly.

The original code execution output of IPython Kernel is also provided below for reference:

entity_name                  metric  deviation
0                 node_cpu_usage_rate  10.995086
1                node_disk_usage_rate   0.095980
2              node_memory_usage_rate   0.346318```
```

In [ ]:
# In[12]:

# Filter for the `node_cpu_usage_rate` anomalies
cpu_anomalies = filtered_node_metrics[filtered_node_metrics['metric'] == 'node_cpu_usage_rate']

# Identify the node with the highest CPU usage deviation
node_with_highest_cpu_deviation = cpu_anomalies.loc[cpu_anomalies['deviation'].idxmax()]

node_with_highest_cpu_deviation

```
Out[12]:
```
The node associated with the `node_cpu_usage_rate` anomaly during the alert time window could not be identified by name, as the `entity_name` field is empty. However, this node experienced the highest CPU usage deviation of 10.9951, with a recorded value of 18.9723, significantly exceeding the global P95 threshold. Further investigation is needed to determine the specific node identity.

The original code execution output of IPython Kernel is also provided below for reference:

time                                             1775313849000000
domain                                                        k8s
entity_set                                               k8s.node
entity_id                                                        
entity_name                                                      
metric                                        node_cpu_usage_rate
value                                                   18.972345
metric_set_id    k8s@metric_set@k8s.metric.high_level_metric_node
service                                                      None
deviation                                               10.995086
Name: 12034, dtype: object```
```